# 08 — Synthetic Clinical Trial Simulator

**Project P.U.L.S.E.** — Predictive Unified Life-sciences Summarization Engine

This notebook simulates the analytical workflow of a clinical trial using **propensity score matching** on the Diabetes 130-US hospital dataset. It demonstrates the core methodology used in **Real-World Evidence (RWE)** studies and **post-market surveillance** — two priority areas for Abbott's Global Data Science & Analytics (GDSA) team.

**Workflow:**
1. Define a synthetic treatment arm from the hospital data (proxy: high-risk HbA1c patients)
2. Match treatment vs control cohorts using 1:1 nearest-neighbour propensity score matching
3. Compare readmission outcomes between arms
4. Visualise time-to-event with a Kaplan-Meier survival curve
5. Test statistical significance with a log-rank test

All data is real clinical data re-purposed for simulation; the treatment assignment and time-to-event variables are synthetic proxies for demonstration purposes.

In [ ]:
# Cell 1 — Setup and data loading
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("data/processed/master_patient_table.parquet")
# Use Diabetes 130 as the "hospital trial population"
trial_df = df[df["source_dataset"] == "diabetes_130"].copy()
trial_df = trial_df.dropna(subset=["age", "gender", "bmi", "diabetes_label"])
print(f"Trial population: {len(trial_df)} patients")
trial_df[["age", "gender", "bmi", "hba1c", "diabetes_label"]].describe().round(2)

## Step 1: Propensity Score Modelling

In a real clinical trial, treatment assignment is randomised. In a **Real-World Evidence** study using observational data, patients self-select into treatment — creating confounding bias. **Propensity Score Matching (PSM)** corrects for this by estimating the probability of receiving treatment given a patient's baseline covariates, then matching treated and untreated patients with similar scores.

Here we define a synthetic treatment arm: patients with **HbA1c ≥ 8.0%** are classified as 'high-risk treated' (proxy for insulin-intensification therapy). A logistic regression model estimates the propensity score — the probability of being in the treatment group — from four clinical baseline features.

In [ ]:
# Cell 2 — Propensity Score Matching
# Simulate a treatment arm (e.g., patients who received insulin) vs control
# In Diabetes 130, use HbA1c > 8 as a proxy for "high-risk, likely treated" group

trial_df["treatment"] = (trial_df["hba1c"] >= 8.0).astype(int)

FEATURES = ["age", "gender", "bmi", "blood_pressure_sys"]
X = trial_df[FEATURES].fillna(trial_df[FEATURES].median())
y = trial_df["treatment"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit propensity score model
ps_model = LogisticRegression(max_iter=500, random_state=42)
ps_model.fit(X_scaled, y)
trial_df["propensity_score"] = ps_model.predict_proba(X_scaled)[:, 1]

print("Propensity score distribution:")
print(trial_df.groupby("treatment")["propensity_score"].describe().round(3))

## Step 2: 1:1 Nearest-Neighbour Matching

With propensity scores estimated, we create a balanced matched cohort by pairing each treated patient with the control patient most similar in propensity score (1:1 nearest-neighbour matching). A well-matched cohort shows similar means across all baseline covariates — this **balance check** is the equivalent of the Table 1 in a clinical trial paper.

After matching, the remaining differences between arms are no longer attributable to measured baseline confounders, allowing us to attribute outcome differences to the treatment itself.

In [ ]:
# Cell 3 — Match treatment and control arms (1:1 nearest neighbor)
from sklearn.neighbors import NearestNeighbors

treated = trial_df[trial_df["treatment"] == 1].copy()
control = trial_df[trial_df["treatment"] == 0].copy()

# Fit nearest neighbor on propensity scores
nn = NearestNeighbors(n_neighbors=1, algorithm="ball_tree")
nn.fit(control[["propensity_score"]])
distances, indices = nn.kneighbors(treated[["propensity_score"]])

matched_control = control.iloc[indices.flatten()].copy()
matched_treated = treated.copy()

print(f"Matched cohort: {len(matched_treated)} treated, {len(matched_control)} control")
print("\nBalance check after matching (means should be close):")
for col in FEATURES:
    t_mean = matched_treated[col].mean()
    c_mean = matched_control[col].mean()
    print(f"  {col}: treated={t_mean:.2f}, control={c_mean:.2f}, diff={abs(t_mean-c_mean):.2f}")

## Step 3: Outcome Analysis

With the matched cohort defined, we compare the primary outcome — **30-day readmission rate** (proxied by the `diabetes_label` column) — between the treatment and control arms.

Three core effect-size metrics are computed:

- **Absolute Risk Reduction (ARR):** The raw difference in outcome rates between arms. Directly clinically interpretable.
- **Relative Risk Reduction (RRR):** ARR expressed as a fraction of the control arm's baseline risk. Used in regulatory submissions.
- **Number Needed to Treat (NNT):** How many patients must receive the treatment for one additional patient to benefit. NNT = 1/ARR. Lower is better; a value above 50 typically indicates the treatment effect is too small to justify routine use.

In [ ]:
# Cell 4 — Outcome analysis: readmission rate comparison
outcome_col = "diabetes_label"  # readmission as outcome proxy

t_outcome = matched_treated[outcome_col].mean()
c_outcome = matched_control[outcome_col].mean()

print("=== Trial Outcome Analysis ===")
print(f"Treatment arm readmission rate: {t_outcome:.3f} ({t_outcome*100:.1f}%)")
print(f"Control arm readmission rate:   {c_outcome:.3f} ({c_outcome*100:.1f}%)")
print(f"Absolute Risk Reduction (ARR):  {c_outcome - t_outcome:.3f}")
print(f"Relative Risk Reduction (RRR):  {(c_outcome - t_outcome) / c_outcome:.3f}")
if c_outcome - t_outcome > 0:
    print(f"Number Needed to Treat (NNT):   {1 / (c_outcome - t_outcome):.1f}")

## Step 4: Kaplan-Meier Survival Curve

The **Kaplan-Meier (KM) estimator** is the standard tool for visualising time-to-event data in clinical trials. It plots the probability that a patient remains event-free (here: readmission-free) at each point in time, and is the cornerstone of survival analysis in oncology, cardiology, and chronic disease trials.

Here we simulate time-to-readmission using an exponential distribution calibrated on the outcome rate — a standard technique for synthetic demonstration when exact dates are unavailable. In a real Abbott trial, these dates would be pulled from the CDISC SDTM `ADTTE` (Analysis Dataset for Time-to-Event) domain.

The shaded bands represent 95% confidence intervals. Separation between curves indicates a treatment effect; overlapping CIs at early time points suggest the effect manifests primarily in longer-term follow-up.

In [ ]:
# Cell 5 — Kaplan-Meier survival curve (time-to-readmission simulation)
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

# Simulate time-to-event: assign random "days to readmission" for demo
np.random.seed(42)
matched_treated["time"] = np.random.exponential(
    scale=120 - 20 * matched_treated[outcome_col], size=len(matched_treated)
).clip(1, 365)
matched_control["time"] = np.random.exponential(
    scale=100 - 20 * matched_control[outcome_col], size=len(matched_control)
).clip(1, 365)

kmf_t = KaplanMeierFitter()
kmf_c = KaplanMeierFitter()

fig, ax = plt.subplots(figsize=(9, 5))
kmf_t.fit(matched_treated["time"], matched_treated[outcome_col], label="Treatment arm")
kmf_c.fit(matched_control["time"], matched_control[outcome_col], label="Control arm")
kmf_t.plot_survival_function(ax=ax, ci_show=True)
kmf_c.plot_survival_function(ax=ax, ci_show=True)
ax.set_title("Kaplan-Meier: Readmission-free survival (simulated trial)")
ax.set_xlabel("Days")
ax.set_ylabel("Survival probability")
plt.tight_layout()
plt.savefig("reports/kaplan_meier.png", dpi=150)
plt.show()
print("Saved to reports/kaplan_meier.png")

## Step 5: Log-Rank Test (Statistical Significance)

The **log-rank test** is the standard non-parametric test for comparing two survival curves. It tests the null hypothesis that the two groups have identical survival functions at all time points.

- **p < 0.05:** The survival curves are statistically different — the treatment effect is likely real, not due to sampling variation
- **p ≥ 0.05:** Insufficient evidence to conclude the curves differ; the trial is inconclusive or underpowered

Note that statistical significance alone is not sufficient for clinical adoption — the **effect size** (ARR, NNT from Step 3) must also be clinically meaningful. A very large trial can detect a statistically significant but clinically trivial difference.

In [ ]:
# Cell 6 — Log Rank Test (statistical significance)
from lifelines.statistics import logrank_test

results = logrank_test(
    matched_treated["time"], matched_control["time"],
    event_observed_A=matched_treated[outcome_col],
    event_observed_B=matched_control[outcome_col],
)

print("=== Log-Rank Test ===")
print(f"Test statistic: {results.test_statistic:.4f}")
print(f"P-value:        {results.p_value:.4f}")
if results.p_value < 0.05:
    print("Result: Statistically significant difference between arms (p < 0.05)")
else:
    print("Result: No statistically significant difference detected")

## Context: CDISC Standards in Real Abbott Trials

In a real Abbott clinical trial, patient data would be structured according to **CDISC SDTM** (Study Data Tabulation Model) standards before any analytical work begins. Key domains relevant to this simulation include:

| SDTM Domain | Relevance to this notebook |
|---|---|
| **DM** (Demographics) | Age, sex, race — the PSM baseline covariates |
| **LB** (Lab Results) | HbA1c, glucose — the treatment proxy and outcome biomarkers |
| **VS** (Vital Signs) | BMI, blood pressure — additional PSM covariates |
| **ADTTE** (Time-to-Event ADaM) | Days-to-readmission — the KM curve input |
| **ADSL** (Subject-Level ADaM) | One row per patient with all derived variables — mirrors our `trial_df` |

The analytical workflow demonstrated here — **cohort definition → propensity score modelling → 1:1 matching → outcome analysis → KM curve → log-rank test** — is the standard sequence used in Abbott's Real-World Evidence studies and is directly applicable to post-market surveillance, health economics, and comparative effectiveness research.

This notebook would form the **SAP (Statistical Analysis Plan) prototype** submitted to the clinical statistician before a real study lock.